## 理解整数：从十进制计数到竖式运算

### 学习目标
- 理解：整数不是一个孤立符号，而是由不同数位上的数字共同表示。
- 会表示：把整数拆成数字列表，例如 `123 -> [1, 2, 3]`。
- 会实现：用列表模拟人类做加法、减法、乘法和除法的竖式过程。
- 能解释：进位、借位、部分积和余数分别解决什么问题。
- 能验证：用 Python 内置整数结果检查自己的按位算法是否正确。

## 十进制整数如何表示？

十进制整数由一串数字组成。每个数字的位置不同，表示的权重也不同。

例如 `123` 可以理解为：

$$
123 = 1 \times 100 + 2 \times 10 + 3 \times 1
$$

| 数位 | 数字 | 权重 | 贡献 |
|---|---:|---:|---:|
| 百位 | 1 | 100 | 100 |
| 十位 | 2 | 10 | 20 |
| 个位 | 3 | 1 | 3 |

在代码中，我们可以先把整数拆成数字列表：`123 -> [1, 2, 3]`。这样后面就能像竖式一样，从个位开始逐位处理。

```{mermaid}
flowchart LR
    A[整数 123] --> B[字符 '1' '2' '3']
    B --> C[数字列表 1, 2, 3]
    C --> D[按数位计算]
```

In [ ]:
def int_to_digits(number):
    """把非负整数拆成数字列表。"""
    text = str(number)
    digits = []

    for ch in text:
        digits.append(int(ch))

    return digits


def digits_to_int(digits):
    """把数字列表还原成整数。"""
    number = 0

    for digit in digits:
        number = number * 10 + digit

    return number


digits = int_to_digits(123)
number = digits_to_int(digits)

print("123 拆成:", digits)
print("还原结果:", number)

## 整数加法：从个位开始进位

竖式加法的核心规则是：

- 从个位开始相加。
- 如果当前位结果大于等于 10，就向高一位进 1。
- 当前位只保留个位数字。

例如 `78 + 45`：个位 `8 + 5 = 13`，写 3，向十位进 1；十位 `7 + 4 + 1 = 12`，写 2，再向百位进 1。

```{mermaid}
flowchart TD
    A[从个位开始] --> B[相加当前位和进位]
    B --> C{结果是否 >= 10}
    C -- 是 --> D[当前位写 sum % 10]
    D --> E[进位 carry = 1]
    C -- 否 --> F[当前位写 sum]
    F --> G[进位 carry = 0]
    E --> H[处理高一位]
    G --> H
```

In [ ]:
def add_digits(left, right):
    i = len(left) - 1
    j = len(right) - 1
    carry = 0
    result = []

    while i >= 0 or j >= 0 or carry > 0:
        if i >= 0:
            a = left[i]
        else:
            a = 0

        if j >= 0:
            b = right[j]
        else:
            b = 0

        total = a + b + carry
        result_digit = total % 10
        carry = total // 10

        result.insert(0, result_digit)
        i = i - 1
        j = j - 1

    return result


left = int_to_digits(78)
right = int_to_digits(45)
answer = add_digits(left, right)

print("数字列表:", left, "+", right)
print("加法结果列表:", answer)
print("加法结果整数:", digits_to_int(answer))

## 整数减法：不够减时向高位借位

本节先只处理“较大的非负整数减较小的非负整数”，例如 `123 - 58`。

竖式减法的核心规则是：

- 从个位开始相减。
- 如果当前位不够减，就向高一位借 1。
- 借来的 1 在当前位等于 10。

```{mermaid}
flowchart TD
    A[从个位开始] --> B[计算当前位 a - borrow]
    B --> C{是否小于 b}
    C -- 是 --> D[当前位加 10 再减 b]
    D --> E[下一位需要借位 borrow = 1]
    C -- 否 --> F[直接相减]
    F --> G[borrow = 0]
```

In [ ]:
def remove_leading_zeros(digits):
    while len(digits) > 1 and digits[0] == 0:
        digits.pop(0)
    return digits


def compare_digits(left, right):
    left = remove_leading_zeros(left[:])
    right = remove_leading_zeros(right[:])

    if len(left) > len(right):
        return 1
    if len(left) < len(right):
        return -1

    for i in range(len(left)):
        if left[i] > right[i]:
            return 1
        if left[i] < right[i]:
            return -1

    return 0


def subtract_digits(left, right):
    assert compare_digits(left, right) >= 0

    i = len(left) - 1
    j = len(right) - 1
    borrow = 0
    result = []

    while i >= 0:
        a = left[i] - borrow
        if j >= 0:
            b = right[j]
        else:
            b = 0

        if a < b:
            a = a + 10
            borrow = 1
        else:
            borrow = 0

        result.insert(0, a - b)
        i = i - 1
        j = j - 1

    return remove_leading_zeros(result)


left = int_to_digits(123)
right = int_to_digits(58)
answer = subtract_digits(left, right)

print("数字列表:", left, "-", right)
print("减法结果列表:", answer)
print("减法结果整数:", digits_to_int(answer))

## 整数乘法：一位一位产生部分积

竖式乘法可以拆成两层：

1. 先计算一个多位数乘一位数。
2. 再把每一位产生的部分积按位置补 0 后相加。

例如 `123 × 45` 可以看成：

$$
123 \times 45 = 123 \times 5 + 123 \times 40
$$

这里的 `40` 来自十位上的 `4`，所以部分积后面要补 1 个 `0`。

In [ ]:
def multiply_by_digit(digits, digit):
    i = len(digits) - 1
    carry = 0
    result = []

    while i >= 0 or carry > 0:
        if i >= 0:
            a = digits[i]
        else:
            a = 0

        total = a * digit + carry
        result.insert(0, total % 10)
        carry = total // 10
        i = i - 1

    return remove_leading_zeros(result)


def multiply_digits(left, right):
    result = [0]
    zeros = 0

    for i in range(len(right) - 1, -1, -1):
        part = multiply_by_digit(left, right[i])

        for _ in range(zeros):
            part.append(0)

        result = add_digits(result, part)
        zeros = zeros + 1

    return remove_leading_zeros(result)


left = int_to_digits(123)
right = int_to_digits(45)
answer = multiply_digits(left, right)

print("数字列表:", left, "*", right)
print("乘法结果列表:", answer)
print("乘法结果整数:", digits_to_int(answer))

## 整数除法：逐步确定商和余数

整数除法要回答两个问题：

- 商是多少？
- 剩下不能继续整除的余数是多少？

例如 `125 ÷ 12 = 10 ... 5`，表示 `125 = 12 × 10 + 5`。

长除法的直觉是：从左到右逐位读入被除数，每次看当前余数里最多能放几个除数。

In [ ]:
def divide_digits(dividend, divisor):
    assert compare_digits(divisor, [0]) > 0

    quotient = []
    remainder = [0]

    for digit in dividend:
        remainder.append(digit)
        remainder = remove_leading_zeros(remainder)

        q_digit = 0
        while compare_digits(remainder, divisor) >= 0:
            remainder = subtract_digits(remainder, divisor)
            q_digit = q_digit + 1

        quotient.append(q_digit)

    return remove_leading_zeros(quotient), remove_leading_zeros(remainder)


dividend = int_to_digits(125)
divisor = int_to_digits(12)
quotient, remainder = divide_digits(dividend, divisor)

print("被除数:", dividend)
print("除数:", divisor)
print("商:", quotient, "=", digits_to_int(quotient))
print("余数:", remainder, "=", digits_to_int(remainder))

## 应用样例：统一检查四种运算

我们写的函数都使用“数字列表”作为接口：

| 函数 | 输入 | 输出 | 说明 |
|---|---|---|---|
| `int_to_digits` | 整数 | 数字列表 | 拆分整数 |
| `digits_to_int` | 数字列表 | 整数 | 还原整数 |
| `add_digits` | 两个数字列表 | 数字列表 | 加法 |
| `subtract_digits` | 两个数字列表 | 数字列表 | 减法，要求左边不小于右边 |
| `multiply_digits` | 两个数字列表 | 数字列表 | 乘法 |
| `divide_digits` | 两个数字列表 | 商和余数 | 除法，要求除数不是 0 |

这些实现不是为了替代 Python 的 `+ - * // %`，而是为了观察运算内部的按位过程。

In [ ]:
a = int_to_digits(256)
b = int_to_digits(37)

sum_answer = add_digits(a, b)
sub_answer = subtract_digits(a, b)
mul_answer = multiply_digits(a, b)
quotient, remainder = divide_digits(a, b)

print("256 + 37 =", digits_to_int(sum_answer))
print("256 - 37 =", digits_to_int(sub_answer))
print("256 * 37 =", digits_to_int(mul_answer))
print("256 // 37 =", digits_to_int(quotient))
print("256 % 37 =", digits_to_int(remainder))

assert digits_to_int(sum_answer) == 256 + 37
assert digits_to_int(sub_answer) == 256 - 37
assert digits_to_int(mul_answer) == 256 * 37
assert digits_to_int(quotient) == 256 // 37
assert digits_to_int(remainder) == 256 % 37

## 适用范围与局限性

为了让算法更容易观察，本节代码做了简化：

- 只处理非负整数。
- 减法要求被减数不小于减数。
- 除法只做整数除法，返回商和余数。
- 为了教学清晰，代码没有追求最高效率。

真实 Python 的整数运算还会处理负数、大整数内存布局、性能优化等问题。本节先把重点放在“竖式为什么成立”。

## 习题

### 1. 概念判断：为什么整数要按数位理解？

为什么 `123` 不能只理解成三个数字 `1`、`2`、`3` 的简单排列？数位权重在其中起什么作用？

<details>
<summary>参考答案</summary>

`123` 中每个数字的位置不同，权重也不同。`1` 在百位表示 100，`2` 在十位表示 20，`3` 在个位表示 3。因此 `123 = 100 + 20 + 3`。如果不看数位，只看数字排列，就无法解释每个数字贡献了多少数值。

</details>

### 2. 概念判断：加法为什么要从个位开始？

竖式加法中，为什么通常从个位开始，而不是从最高位开始？

<details>
<summary>参考答案</summary>

因为低位相加可能产生进位，这个进位会影响高一位。如果先算最高位，后面低位产生进位时，还要回头修改高位。从个位开始可以把进位自然传给下一位。

</details>

### 3. 概念判断：除法中的余数必须满足什么条件？

如果 `a ÷ b = q ... r`，余数 `r` 应该满足什么条件？为什么？

<details>
<summary>参考答案</summary>

余数必须满足 `0 <= r < b`。如果 `r >= b`，说明剩下的部分还能再分出至少一个 `b`，商还可以继续增加，因此当前商不是最大的整数商。

</details>

### 4. 代码填空：整数与数字列表互相转换

请补全代码，把 `408` 拆成数字列表，再还原成整数。

In [ ]:
number = 408

# Todo：请填空，把整数拆成数字列表
digits = ...

# Todo：请填空，把数字列表还原成整数
back_to_number = ...

if digits is ... or back_to_number is ...:
    print("请完成题目后再运行")
else:
    assert digits == [4, 0, 8]
    assert back_to_number == 408
    print("第 4 题验证通过")

<details>
<summary>第 4 题参考答案</summary>

```python
digits = int_to_digits(number)
back_to_number = digits_to_int(digits)
```

</details>

### 5. 代码填空：用按位加法计算结果

请补全代码，用 `add_digits` 计算 `389 + 76`。

In [ ]:
left = int_to_digits(389)
right = int_to_digits(76)

# Todo：请填空，调用按位加法函数
answer_digits = ...

# Todo：请填空，把结果列表还原成整数
answer = ...

if answer_digits is ... or answer is ...:
    print("请完成题目后再运行")
else:
    assert answer_digits == [4, 6, 5]
    assert answer == 465
    print("第 5 题验证通过")

<details>
<summary>第 5 题参考答案</summary>

```python
answer_digits = add_digits(left, right)
answer = digits_to_int(answer_digits)
```

</details>

### 6. 代码填空：用除法得到商和余数

请补全代码，用 `divide_digits` 计算 `100 ÷ 9` 的商和余数。

In [ ]:
dividend = int_to_digits(100)
divisor = int_to_digits(9)

# Todo：请填空，调用除法函数，得到商列表和余数列表
quotient_digits, remainder_digits = ..., ...

# Todo：请填空，把商和余数还原成整数
quotient = ...
remainder = ...

if quotient_digits is ... or remainder_digits is ... or quotient is ... or remainder is ...:
    print("请完成题目后再运行")
else:
    assert quotient == 11
    assert remainder == 1
    print("第 6 题验证通过")

<details>
<summary>第 6 题参考答案</summary>

```python
quotient_digits, remainder_digits = divide_digits(dividend, divisor)
quotient = digits_to_int(quotient_digits)
remainder = digits_to_int(remainder_digits)
```

</details>